# EdgeElevate API Integration Tests

This notebook tests each API integration separately before building the full workflow.

## APIs to Test:
1. **Peec AI** - Competitor Analysis
2. **Tavily** - Review Scraping (Trustpilot, G2)
3. **Q-Context** - Insight Structuring
4. **Hera** - Video Generation
5. **OpenRouter** - LLM Operations

## Setup and Configuration

In [2]:
import os
import json
import requests
from dotenv import load_dotenv
from typing import Dict, List, Any

# Load environment variables
load_dotenv()

# API Keys
PEEC_API_KEY = os.getenv("PEEC_API_KEY")
QCONTEXT_API_KEY = os.getenv("QCONTEXT_API_KEY")
HERA_API_KEY = os.getenv("HERA_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")

# API Base URLs
PEEC_BASE_URL = "https://api.peec.ai/customer/v1"
QCONTEXT_BASE_URL = "https://api.qontext.ai/v1"
HERA_BASE_URL = "https://api.hera.video"
TAVILY_BASE_URL = "https://api.tavily.com"
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"

# Helper function for pretty printing
def pretty_print(title: str, data: Any):
    print(f"\n{'='*20} {title} {'='*20}")
    print(json.dumps(data, indent=2))

print("✅ Environment loaded successfully!")
print(f"Peec API Key: {'✓' if PEEC_API_KEY else '✗'}")
print(f"Q-Context API Key: {'✓' if QCONTEXT_API_KEY else '✗'}")
print(f"Hera API Key: {'✓' if HERA_API_KEY else '✗'}")
print(f"Tavily API Key: {'✓' if TAVILY_API_KEY else '✗'}")
print(f"OpenRouter API Key: {'✓' if OPENROUTER_API_KEY else '✗'}")

✅ Environment loaded successfully!
Peec API Key: ✗
Q-Context API Key: ✗
Hera API Key: ✗
Tavily API Key: ✓
OpenRouter API Key: ✗


## Test 1: Peec AI - Competitor Analysis

**Purpose**: Identify competitors and extract positioning data

**API Docs**: https://docs.peec.ai/api

In [ ]:
def test_peec_competitor_analysis(startup_name: str):
    """
    Test Peec AI competitor analysis
    Note: This requires Enterprise tier access
    """
    try:
        # First, we need to create a brand/project
        headers = {
            "Authorization": f"Bearer {PEEC_API_KEY}",
            "Content-Type": "application/json"
        }
        
        # Example: List existing brands/projects
        response = requests.get(
            f"{PEEC_BASE_URL}/brands",
            headers=headers
        )
        
        if response.status_code == 200:
            brands_data = response.json()
            pretty_print("Peec AI - Brands List", brands_data)
            return brands_data
        else:
            print(f"❌ Peec API Error: {response.status_code}")
            print(f"Response: {response.text}")
            return None
            
    except Exception as e:
        print(f"❌ Error testing Peec AI: {str(e)}")
        return None

# Test with a startup name
startup_name = "Nothing Phone"
peec_result = test_peec_competitor_analysis(startup_name)

## Test 2: Tavily - Review Scraping

**Purpose**: Scrape reviews from Trustpilot, G2, and other sources

**API Docs**: https://docs.tavily.com

In [3]:
def test_tavily_review_search(startup_name: str, source: str = "trustpilot"):
    """
    Test Tavily search API for finding reviews
    """
    try:
        headers = {
            "Content-Type": "application/json"
        }
        
        # Search for reviews
        payload = {
            "api_key": TAVILY_API_KEY,
            "query": f"{startup_name} {source} reviews customer feedback",
            "search_depth": "advanced",
            "include_domains": [f"{source}.com", "g2.com"],
            "max_results": 5
        }
        
        response = requests.post(
            f"{TAVILY_BASE_URL}/search",
            headers=headers,
            json=payload
        )
        
        if response.status_code == 200:
            search_results = response.json()
            pretty_print(f"Tavily - {startup_name} Reviews", search_results)
            return search_results
        else:
            print(f"❌ Tavily API Error: {response.status_code}")
            print(f"Response: {response.text}")
            return None
            
    except Exception as e:
        print(f"❌ Error testing Tavily: {str(e)}")
        return None

# Test review scraping
tavily_result = test_tavily_review_search("Nothing Phone", "trustpilot")
print("tavily result", tavily_result)


==================== Tavily - Nothing Phone Reviews ====================
{
  "query": "Nothing Phone trustpilot reviews customer feedback",
  "follow_up_questions": null,
  "answer": null,
  "images": [],
  "results": [
    {
      "url": "https://ca.trustpilot.com/review/nothing.tech?page=4",
      "title": "Read Customer Service Reviews of nothing.tech | 4 of 25 - Trustpilot",
      "content": "I have nothing 2a, since last update. Apps close prematurely, internet and WiFi connectivity issues and battery drain is ridiculous.",
      "score": 0.99986446,
      "raw_content": null
    },
    {
      "url": "https://www.trustpilot.com/review/nothing.tech",
      "title": "Read Customer Service Reviews of nothing.tech - Trustpilot",
      "content": "I have nothing 2a, since last update. Apps close prematurely, internet and WiFi connectivity issues and battery drain is ridiculous. I only needed to charge",
      "score": 0.9998287,
      "raw_content": null
    },
    {
      "url": "ht

## Test 3: Q-Context - Insight Structuring

**Purpose**: Ingest data and retrieve structured insights

**API Docs**: https://docs.qontext.ai

In [ ]:
def test_qcontext_ingestion_and_retrieval(data_to_ingest: str, query: str):
    """
    Test Q-Context API for data ingestion and context retrieval
    """
    try:
        headers = {
            "X-API-Key": QCONTEXT_API_KEY,
            "Content-Type": "application/json"
        }
        
        # Step 1: Ingest unstructured data
        print("\n📥 Ingesting data into Q-Context...")
        ingest_payload = {
            "text": data_to_ingest
        }
        
        ingest_response = requests.post(
            f"{QCONTEXT_BASE_URL}/ingestion/unstructured",
            headers=headers,
            json=ingest_payload
        )
        
        if ingest_response.status_code == 201:
            ingest_result = ingest_response.json()
            pretty_print("Q-Context - Ingestion Result", ingest_result)
        else:
            print(f"❌ Ingestion Error: {ingest_response.status_code}")
            print(f"Response: {ingest_response.text}")
            return None
        
        # Step 2: Retrieve context based on query
        print("\n🔍 Retrieving structured insights...")
        retrieval_payload = {
            "prompt": query,
            "includeSources": True
        }
        
        retrieval_response = requests.post(
            f"{QCONTEXT_BASE_URL}/retrieval",
            headers=headers,
            json=retrieval_payload
        )
        
        if retrieval_response.status_code == 200:
            retrieval_result = retrieval_response.json()
            pretty_print("Q-Context - Retrieved Insights", retrieval_result)
            return retrieval_result
        else:
            print(f"❌ Retrieval Error: {retrieval_response.status_code}")
            print(f"Response: {retrieval_response.text}")
            return None
            
    except Exception as e:
        print(f"❌ Error testing Q-Context: {str(e)}")
        return None

# Test data ingestion and retrieval
sample_competitor_data = """
Nothing Phone competes with Apple iPhone, Samsung Galaxy, and Google Pixel.
Strengths: Unique transparent design, competitive pricing, clean Android experience.
Weaknesses: Limited availability, smaller ecosystem, camera quality behind leaders.
Market positioning: Premium design at mid-range pricing for tech enthusiasts.
"""

qcontext_result = test_qcontext_ingestion_and_retrieval(
    data_to_ingest=sample_competitor_data,
    query="What are the key strengths and weaknesses of Nothing Phone?"
)

## Test 4: Hera - Video Generation

**Purpose**: Generate video scripts and create promotional videos

**API Docs**: https://docs.hera.video

In [ ]:
def test_hera_video_generation(prompt: str):
    """
    Test Hera API for video generation
    """
    try:
        headers = {
            "x-api-key": HERA_API_KEY,
            "Content-Type": "application/json"
        }
        
        # Create video job
        print("\n🎬 Creating video generation job...")
        video_payload = {
            "prompt": prompt,
            "formats": ["mp4"],
            "aspectRatio": "16:9"
        }
        
        create_response = requests.post(
            f"{HERA_BASE_URL}/videos",
            headers=headers,
            json=video_payload
        )
        
        if create_response.status_code in [200, 201]:
            job_result = create_response.json()
            pretty_print("Hera - Video Job Created", job_result)
            
            # Get video ID from response
            video_id = job_result.get("id") or job_result.get("video_id")
            
            if video_id:
                # Check job status
                print(f"\n📊 Checking video status for ID: {video_id}...")
                status_response = requests.get(
                    f"{HERA_BASE_URL}/videos/{video_id}",
                    headers=headers
                )
                
                if status_response.status_code == 200:
                    status_result = status_response.json()
                    pretty_print("Hera - Video Status", status_result)
                    return status_result
            
            return job_result
        else:
            print(f"❌ Hera API Error: {create_response.status_code}")
            print(f"Response: {create_response.text}")
            return None
            
    except Exception as e:
        print(f"❌ Error testing Hera: {str(e)}")
        return None

# Test video generation
video_prompt = """
Create a professional product video for Nothing Phone showcasing:
1. Unique transparent design
2. Premium features at competitive pricing
3. Clean Android experience
4. Target audience: Tech enthusiasts and design lovers
"""

hera_result = test_hera_video_generation(video_prompt)

## Test 5: OpenRouter - LLM Operations

**Purpose**: Use cheaper LLM models for various text generation tasks

**API Docs**: https://openrouter.ai/docs

In [ ]:
def test_openrouter_llm(prompt: str, model: str = "meta-llama/llama-3.2-3b-instruct"):
    """
    Test OpenRouter API for LLM operations
    Using cheaper models like Llama 3.2 3B for cost efficiency
    """
    try:
        headers = {
            "Authorization": f"Bearer {OPENROUTER_API_KEY}",
            "Content-Type": "application/json"
        }
        
        payload = {
            "model": model,
            "messages": [
                {
                    "role": "user",
                    "content": prompt
                }
            ]
        }
        
        response = requests.post(
            f"{OPENROUTER_BASE_URL}/chat/completions",
            headers=headers,
            json=payload
        )
        
        if response.status_code == 200:
            result = response.json()
            pretty_print(f"OpenRouter - {model}", result)
            
            # Extract the actual response
            if "choices" in result and len(result["choices"]) > 0:
                message = result["choices"][0]["message"]["content"]
                print("\n📝 Generated Response:")
                print(message)
            
            return result
        else:
            print(f"❌ OpenRouter API Error: {response.status_code}")
            print(f"Response: {response.text}")
            return None
            
    except Exception as e:
        print(f"❌ Error testing OpenRouter: {str(e)}")
        return None

# Test LinkedIn post generation
linkedin_prompt = """
Generate 3 LinkedIn posts for Nothing Phone based on this positioning:
- Unique transparent design sets it apart from competitors
- Premium features at mid-range pricing
- Clean Android experience without bloatware
- Target audience: Tech enthusiasts and design-conscious users

Post 1: Market insight/gap opportunity
Post 2: Founder/vision narrative
Post 3: Product-led feature comparison

Keep each post under 200 words and engaging.
"""

openrouter_result = test_openrouter_llm(linkedin_prompt)

## Summary: API Integration Status

Run this cell to see which APIs are working

In [ ]:
print("\n" + "="*60)
print("API INTEGRATION TEST SUMMARY")
print("="*60)

results = {
    "Peec AI (Competitor Analysis)": peec_result is not None,
    "Tavily (Review Scraping)": tavily_result is not None,
    "Q-Context (Insight Structuring)": qcontext_result is not None,
    "Hera (Video Generation)": hera_result is not None,
    "OpenRouter (LLM Operations)": openrouter_result is not None
}

for api_name, status in results.items():
    status_icon = "✅" if status else "❌"
    print(f"{status_icon} {api_name}: {'Working' if status else 'Failed'}")

working_count = sum(results.values())
total_count = len(results)
print(f"\n📊 Overall: {working_count}/{total_count} APIs working")

if working_count == total_count:
    print("\n🎉 All APIs are ready! You can proceed to build the full workflow.")
elif working_count > 0:
    print("\n⚠️ Some APIs need configuration. Check the error messages above.")
else:
    print("\n❌ No APIs are working. Please check your API keys in the .env file.")